# Course 1 — M08 Statistics — Visual Simulation Notebook RC1

Use this only **after** the matching learner-book lesson. It is a visual companion, not a substitute for the workbook or your explanation.

Rule: change one parameter, rerun, then explain **what changed and what did not**. Every figure is separate.


In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)
print("numpy", np.__version__)
print("pandas", pd.__version__)


## STAT1 — Center, spread and outlier sensitivity

In [ ]:
x=np.array([18,22,24,25,27,28,29,30,31,31,32,33,34,35,36,37,38,39,41,44,52,145],dtype=float)
print("n",len(x),"mean",round(x.mean(),2),"median",np.median(x),"sample SD",round(x.std(ddof=1),2))
q1,q3=np.percentile(x,[25,75],method="linear")
print("Q1",q1,"Q3",q3,"IQR",q3-q1,"upper fence",q3+1.5*(q3-q1))
plt.figure()
plt.hist(x,bins=10)
plt.title("STAT1 — Delivery minutes")
plt.xlabel("Minutes")
plt.ylabel("Count")
plt.show()


## STAT2 — Distribution shape and z-scores

In [ ]:
service=np.array([8,9,10,10,11,11,12,12,12,13,13,14,14,15,16,17,18,19,20,22],dtype=float)
purchase=np.array([120,135,150,165,180,195,210,230,250,275,300,340,390,460,550,700,900,1300,2400,5800],dtype=float)
m=service.mean(); sd=service.std(ddof=1)
print("service mean/median/sd",round(m,2),np.median(service),round(sd,2))
print("z(8)",round((8-m)/sd,2),"z(22)",round((22-m)/sd,2))
print("purchase mean/median",round(purchase.mean(),2),np.median(purchase))
plt.figure()
plt.hist(service,bins=8)
plt.title("STAT2 — Near-symmetric service time")
plt.xlabel("Minutes"); plt.ylabel("Count"); plt.show()
plt.figure()
plt.hist(purchase,bins=10)
plt.title("STAT2 — Right-skewed purchase value")
plt.xlabel("Purchase value"); plt.ylabel("Count"); plt.show()


## STAT3 — Sampling error versus sampling bias

In [ ]:
population=np.arange(1,10001)
rng=np.random.default_rng(42)
random_sample=rng.choice(population,400,replace=False)
biased_frame=population[:2500]
biased_sample=rng.choice(biased_frame,400,replace=False)
print("population mean",population.mean())
print("random sample mean",round(random_sample.mean(),2))
print("biased-frame sample mean",round(biased_sample.mean(),2))
print("The biased frame systematically excludes 75% of the population; larger n inside that frame does not repair coverage.")


## STAT4 — Sampling distributions, SE and CLT

In [ ]:
population=np.random.lognormal(mean=3.8,sigma=0.6,size=200000)
print("population SD",round(population.std(ddof=1),2))
for n in [10,40,160]:
    means=np.array([np.mean(np.random.choice(population,n,replace=True)) for _ in range(1000)])
    print("n",n,"mean of sample means",round(means.mean(),2),"SD of sample means",round(means.std(ddof=1),2))
    plt.figure()
    plt.hist(means,bins=30)
    plt.title(f"STAT4 — Sampling distribution of mean, n={n}")
    plt.xlabel("Sample mean"); plt.ylabel("Count"); plt.show()


## STAT5 — Confidence interval procedure

In [ ]:
mu=50; sigma=12; n=64; reps=1000; covered=0
for _ in range(reps):
    sample=np.random.normal(mu,sigma,n)
    mean=sample.mean()
    se=sample.std(ddof=1)/math.sqrt(n)
    lo,hi=mean-1.96*se,mean+1.96*se
    covered += lo <= mu <= hi
print("Approximate repeated-procedure coverage",covered/reps)
print("After one interval is calculated, describe the procedure/compatible values rather than assigning 95% probability to a fixed parameter.")


## STAT6 — Two-proportion hypothesis test

In [ ]:
def two_prop(nc,cc,nt,ct):
    pc,pt=cc/nc,ct/nt
    pooled=(cc+ct)/(nc+nt)
    se0=math.sqrt(pooled*(1-pooled)*(1/nc+1/nt))
    z=(pt-pc)/se0
    p=math.erfc(abs(z)/math.sqrt(2))
    return pc,pt,z,p
pc,pt,z,p=two_prop(10000,500,10000,540)
print("control",pc,"treatment",pt,"z",z,"two-sided p",p)
print("p-value is not the probability that H0 is true.")


## STAT7 — Effect size versus statistical significance

In [ ]:
cases=[("moderate n",10000,500,10000,540),("huge n",1000000,50000,1000000,50100)]
for label,nc,cc,nt,ct in cases:
    pc,pt,z,p=two_prop(nc,cc,nt,ct)
    abs_pp=(pt-pc)*100
    rel=(pt-pc)/pc*100
    print(label,"absolute lift pp",round(abs_pp,4),"relative lift %",round(rel,4),"p",p)
print("Business value still needs a predeclared worthwhile-effect threshold and guardrails.")


## STAT8 — Correlation and confounding

In [ ]:
stores=300
size=np.random.lognormal(4.5,.45,stores)
ad_spend=size*2 + np.random.normal(0,30,stores)
sales=size*20 + np.random.normal(0,300,stores)
print("corr(ad_spend,sales)",np.corrcoef(ad_spend,sales)[0,1])
plt.figure()
plt.scatter(ad_spend,sales,s=12)
plt.xlabel("Ad spend")
plt.ylabel("Sales")
plt.title("STAT8 — Correlation can share a common cause")
plt.show()
print("Here store size influences both variables, so correlation alone cannot identify a causal effect of ad spend.")


## STAT9 — A/B decision discipline

In [ ]:
nc,cc,nt,ct=1000,120,1000,150
pc,pt,z,p=two_prop(nc,cc,nt,ct)
se_ci=math.sqrt(pc*(1-pc)/nc + pt*(1-pt)/nt)
lo=((pt-pc)-1.96*se_ci)*100
hi=((pt-pc)+1.96*se_ci)*100
support_c=0.07; support_t=0.072
print("absolute lift pp",(pt-pc)*100)
print("relative lift %",(pt-pc)/pc*100)
print("p",p,"95% CI pp",(lo,hi))
print("support change pp",(support_t-support_c)*100)
print("Now compare with the predeclared business threshold and guardrail before deciding.")


## Final explain-back

For each STAT lesson, write one sentence answering:
1. What parameter did I change?
2. What changed in the output?
3. What did **not** become justified just because the number/plot changed?

Restart Kernel → Run All before using this notebook as evidence.
